# CIFAR-10 & Transfer Learning

## From MNIST to CIFAR-10

| | MNIST | CIFAR-10 |
|---|---|---|
| Images | 28×28 grayscale | 32×32 RGB |
| Classes | 10 digits | 10 object categories |
| Difficulty | Easy (~99% with CNN) | Harder (~93% with ResNet) |

## What is Transfer Learning?

Training deep CNNs from scratch needs millions of images and days of GPU time. Transfer learning shortcuts this:

1. Take a network **pretrained on ImageNet** (1.2M images, 1000 classes) — it has already learned edges, textures, shapes
2. **Freeze** the backbone (don't update its weights)
3. **Replace** the final classification head with one for your task
4. **Fine-tune** only the head (or gradually unfreeze the backbone)

Result: near state-of-the-art on CIFAR-10 in minutes, not hours.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print('device:', DEVICE)

CIFAR_CLASSES = ['airplane','automobile','bird','cat','deer',
                 'dog','frog','horse','ship','truck']

## 1. Data — with ImageNet normalization

Pretrained models were trained with ImageNet stats — use those same mean/std to normalize CIFAR-10 inputs, otherwise the backbone's learned filters see out-of-distribution inputs.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),          # data augmentation
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = datasets.CIFAR10(root='./data', train=True,  download=True, transform=train_transform)
test_ds  = datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)}  Test: {len(test_ds)}')
print(f'Image shape: {train_ds[0][0].shape}')  # (3, 32, 32)

In [ ]:
# Visualize samples (unnormalize for display)
def unnormalize(img):
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    std  = torch.tensor(IMAGENET_STD).view(3,1,1)
    return (img * std + mean).clamp(0, 1)

imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(unnormalize(imgs[i]).permute(1, 2, 0))
    ax.set_title(CIFAR_CLASSES[labels[i]], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Strategy A — Feature Extraction (frozen backbone)

Freeze all backbone weights. Only the new classifier head trains.  
Fastest option — good when your dataset is small.

In [ ]:
# Load pretrained ResNet-18 (11M params, trained on ImageNet)
backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze everything
for param in backbone.parameters():
    param.requires_grad = False

# Replace the final FC layer (was 512 → 1000 for ImageNet)
in_features = backbone.fc.in_features   # 512
backbone.fc = nn.Linear(in_features, 10)  # 512 → 10 for CIFAR-10
# Only backbone.fc has requires_grad=True now

backbone = backbone.to(DEVICE)

trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
total     = sum(p.numel() for p in backbone.parameters())
print(f'Trainable: {trainable:,} / {total:,} params  ({100*trainable/total:.1f}%)')

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        correct += (model(xb).argmax(1) == yb).sum().item()
        total   += yb.size(0)
    return correct / total


criterion = nn.CrossEntropyLoss()
# Only pass the new head's parameters to the optimizer
optimizer = torch.optim.Adam(backbone.fc.parameters(), lr=1e-3)

print('Feature extraction (frozen backbone):')
for epoch in range(5):
    loss = train_epoch(backbone, train_loader, optimizer, criterion)
    acc  = evaluate(backbone, test_loader)
    print(f'  Epoch {epoch+1}  loss={loss:.4f}  test_acc={acc:.4f}')

## 3. Strategy B — Fine-tuning (unfreeze backbone, lower LR)

After the head converges, unfreeze the backbone and train everything at a **much smaller** learning rate.  
The backbone already has good features — a large LR would destroy them.

Use **differential learning rates**: lower LR for backbone, higher LR for head.

In [ ]:
# Unfreeze all backbone layers
for param in backbone.parameters():
    param.requires_grad = True

# Differential LRs: backbone gets 10× smaller LR than the head
backbone_params = [p for name, p in backbone.named_parameters() if 'fc' not in name]
head_params     = backbone.fc.parameters()

optimizer = torch.optim.Adam([
    {'params': backbone_params, 'lr': 1e-4},   # backbone — tiny LR
    {'params': head_params,     'lr': 1e-3},   # head — normal LR
])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

print('Fine-tuning (unfrozen backbone, differential LRs):')
for epoch in range(10):
    loss = train_epoch(backbone, train_loader, optimizer, criterion)
    acc  = evaluate(backbone, test_loader)
    scheduler.step()
    print(f'  Epoch {epoch+1}  loss={loss:.4f}  test_acc={acc:.4f}')

## 4. What's Inside ResNet-18?

In [ ]:
# Print the architecture
print(backbone)

# Key idea: residual (skip) connections
# output = F(x) + x
# If a layer is useless, gradients can flow unchanged through the skip
# This lets you train much deeper networks without vanishing gradients

In [ ]:
# Visualize learned filters from the first conv layer
first_conv = backbone.conv1.weight.data.cpu()  # (64, 3, 7, 7)

# Normalize each filter to [0,1] for display
filters = first_conv - first_conv.min()
filters = filters / filters.max()

fig, axes = plt.subplots(4, 16, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(filters[i].permute(1, 2, 0).clamp(0, 1))
    ax.axis('off')
plt.suptitle('First conv layer filters (learned on ImageNet)', fontsize=10)
plt.tight_layout()
plt.show()

## 5. Visualize Predictions

In [ ]:
backbone.eval()
imgs, labels = next(iter(test_loader))
with torch.no_grad():
    preds = backbone(imgs.to(DEVICE)).argmax(1).cpu()

fig, axes = plt.subplots(2, 10, figsize=(16, 4))
for i in range(10):
    for row in range(2):
        idx = i + row * 10
        ax = axes[row, i]
        ax.imshow(unnormalize(imgs[idx]).permute(1, 2, 0))
        color = 'green' if preds[idx] == labels[idx] else 'red'
        ax.set_title(f'{CIFAR_CLASSES[preds[idx]]}', fontsize=7, color=color)
        ax.axis('off')
plt.suptitle('Green = correct, Red = wrong', fontsize=9)
plt.tight_layout()
plt.show()

## Summary

| Strategy | When | LR |
|---|---|---|
| Feature extraction | Small dataset, limited GPU | Only train head |
| Full fine-tuning | Larger dataset, more compute | Differential: backbone 10× lower than head |

**Rule of thumb**: always start with feature extraction, then unfreeze.

**Next steps**:
- Try `ResNet-50` or `EfficientNet` via `torchvision.models` — same code, swap the model line
- Add `BatchNorm` freeze during feature extraction (call `model.eval()` on bn layers after `model.train()`)
- Try on your own image dataset with `datasets.ImageFolder`